#Initialization

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

#Read Bronze table

In [0]:
df = spark.table("pcat.bronze.products")

In [0]:
df.show(10)

#Silver Transformations

##Drop Duplicates

In [0]:
duplicate_data = df.groupBy("product_id").count().filter(F.col("count") > 1)
display(duplicate_data)

In [0]:
df = df.dropDuplicates(["product_id"])

##Trim spaces

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

##Fix Name casing

In [0]:
df.select("category").distinct().show()

In [0]:
df = df.withColumn(
    "category",
    F.initcap(F.col("category"))
    )
    

##Fix Spelling for Protien

In [0]:
df = (
    df
    .withColumn(
        "product_name",
        F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein")
    )
    .withColumn(
        "category",
        F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
    )
)

##Check dataframe

In [0]:
display(df.limit(5))

## Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df = (
    df
     .withColumn(
         "division",
         F.when(F.col("category") == "Energy Bars",        "Nutrition Bars")
         .when(F.col("category") == "Protein Bars",       "Nutrition Bars")
         .when(F.col("category") == "Granola & Cereals",  "Breakfast Foods")
         .when(F.col("category") == "Recovery Dairy",     "Dairy & Recovery")
         .when(F.col("category") == "Healthy Snacks",     "Healthy Snacks")
         .when(F.col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
         .otherwise("Other")
     )
)

df = df.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)

df = (
    df
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    .withColumnRenamed("product_name", "product")
)

##Check Dataframe

In [0]:
df = df.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")
df.show(truncate=False)

#Writing Silver Table

In [0]:
df.write \
  .format("delta") \
  .option("delta.enableChangeDataFeed", "true") \
  .option("mergeSchema", "true") \
  .mode("overwrite") \
  .saveAsTable("pcat.silver.products")

##Sanity checks of silver _table_

In [0]:
%sql
SELECT * FROM pcat.silver.products LIMIT 10